# ROC-AUC and Precision-Recall Curves — Built from Scratch

Both curves answer the same question — *how good is my classifier?* — but they tell fundamentally different stories, and choosing the wrong one can make a terrible model look great.

---

## Prerequisites: The Confusion Matrix

Every binary classifier at a fixed threshold produces four outcomes:

```
                    Predicted
                  Positive  Negative
Actual Positive │   TP    │   FN   │  ← all real positives
Actual Negative │   FP    │   TN   │  ← all real negatives
```

| Symbol | Name | Formula | Plain English |
|---|---|---|---|
| **TPR** | True Positive Rate / Recall / Sensitivity | TP / (TP+FN) | Of all real positives, how many did we catch? |
| **FPR** | False Positive Rate / Fall-out | FP / (FP+TN) | Of all real negatives, how many did we wrongly flag? |
| **Precision** | Positive Predictive Value | TP / (TP+FP) | Of everything we called positive, how many really were? |
| **Recall** | = TPR | TP / (TP+FN) | Same as TPR |

---

## The Threshold Problem

A classifier outputs a **score** (e.g. softmax probability), not a hard label.  
We choose a threshold τ to convert scores to labels:

```
ŷ = 1  if score ≥ τ
ŷ = 0  if score < τ
```

Different τ values produce different (FPR, TPR) and (Recall, Precision) operating points.  
**Curves plot all operating points simultaneously** — they show classifier quality independent of any single threshold.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.grid': True,
    'grid.alpha': 0.3, 'font.size': 11
})
rng = np.random.default_rng(42)

# ── From-scratch implementations ─────────────────────────────────────

def confusion_at_threshold(y_true, scores, threshold):
    """Return (TP, FP, TN, FN) for a given threshold."""
    pred = (scores >= threshold).astype(int)
    TP = int(((pred == 1) & (y_true == 1)).sum())
    FP = int(((pred == 1) & (y_true == 0)).sum())
    TN = int(((pred == 0) & (y_true == 0)).sum())
    FN = int(((pred == 0) & (y_true == 1)).sum())
    return TP, FP, TN, FN


def roc_curve_scratch(y_true, scores):
    """
    Build ROC curve by sweeping threshold over all unique score values.
    Returns (fpr_array, tpr_array, thresholds).
    """
    thresholds = np.sort(np.unique(scores))[::-1]  # descending
    # Add a point for threshold above max score (all predicted negative)
    thresholds = np.concatenate([[thresholds[0] + 1e-9], thresholds])

    P = (y_true == 1).sum()  # total positives
    N = (y_true == 0).sum()  # total negatives

    fprs, tprs = [], []
    for tau in thresholds:
        TP, FP, TN, FN = confusion_at_threshold(y_true, scores, tau)
        tprs.append(TP / P if P > 0 else 0.0)
        fprs.append(FP / N if N > 0 else 0.0)

    return np.array(fprs), np.array(tprs), thresholds


def pr_curve_scratch(y_true, scores):
    """
    Build PR curve by sweeping threshold over all unique score values.
    Returns (recall_array, precision_array, thresholds).
    """
    thresholds = np.sort(np.unique(scores))[::-1]
    thresholds = np.concatenate([[thresholds[0] + 1e-9], thresholds])

    recalls, precisions = [], []
    for tau in thresholds:
        TP, FP, TN, FN = confusion_at_threshold(y_true, scores, tau)
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        precision = TP / (TP + FP) if (TP + FP) > 0 else 1.0  # convention
        recalls.append(recall)
        precisions.append(precision)

    return np.array(recalls), np.array(precisions), thresholds


def auc_trapezoid(x, y):
    """Area under curve using the trapezoidal rule."""
    # Sort by x for proper integration
    order = np.argsort(x)
    return float(np.trapezoid(y[order], x[order]))


print("All helper functions defined.")

---
## Part 1 — The ROC Curve

**R**eceiver **O**perating **C**haracteristic curve — plots TPR (y-axis) vs FPR (x-axis) as we sweep threshold τ from 1 → 0.

```
τ = 1.0  → predict everything negative → TPR=0, FPR=0  (bottom-left)
τ → 0.0  → predict everything positive → TPR=1, FPR=1  (top-right)
```

### The Three Reference Classifiers

| Classifier | Curve | AUC |
|---|---|---|
| Perfect | Top-left corner → (0,1) | 1.0 |
| Random | Diagonal (y = x) | 0.5 |
| Worst possible | Bottom-right corner → (1,0) | 0.0 |

**AUC** = Area Under the ROC Curve. Higher is better.

In [ ]:
# ── Generate three classifier scenarios ──────────────────────────────
N = 1000
y_true = (rng.random(N) > 0.5).astype(int)

# Good classifier: scores are well-separated by class
scores_good = np.where(y_true == 1,
    rng.normal(0.70, 0.15, N),
    rng.normal(0.30, 0.15, N)
).clip(0, 1)

# Random classifier: scores are independent of true label
scores_random = rng.random(N)

# Bad classifier: scores are ANTI-correlated with true label
scores_bad = np.where(y_true == 1,
    rng.normal(0.30, 0.15, N),
    rng.normal(0.70, 0.15, N)
).clip(0, 1)

# Compute ROC curves
scenarios = [
    ('Good classifier',   scores_good,   'steelblue'),
    ('Random classifier', scores_random, 'gray'),
    ('Bad classifier',    scores_bad,    'crimson'),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for label, scores, color in scenarios:
    fpr, tpr, _ = roc_curve_scratch(y_true, scores)
    auc = auc_trapezoid(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, linewidth=2.5, label=f'{label}  AUC={auc:.3f}')

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random baseline (AUC=0.5)')
axes[0].fill_between([0, 1], [0, 1], [1, 1], alpha=0.05, color='green', label='Good region')
axes[0].fill_between([0, 1], [0, 0], [0, 1], alpha=0.05, color='red',   label='Bad region')
axes[0].set_xlabel('FPR  (False Positive Rate)')
axes[0].set_ylabel('TPR  (True Positive Rate / Recall)')
axes[0].set_title('ROC Curves')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)

# Score distributions
bins = np.linspace(0, 1, 40)
for scores, label, color, alpha in [
    (scores_good[y_true==1], 'Positive class (good clf)', 'green',  0.6),
    (scores_good[y_true==0], 'Negative class (good clf)', 'red',    0.6),
    (scores_bad[y_true==1],  'Positive class (bad clf)',  'green',  0.25),
    (scores_bad[y_true==0],  'Negative class (bad clf)',  'red',    0.25),
]:
    ls = '-' if 'good' in label else '--'
    axes[1].hist(scores, bins=bins, alpha=alpha, color=color, label=label, 
                 histtype='stepfilled', linewidth=1.5)

axes[1].set_title('Score Distributions by Class')
axes[1].set_xlabel('Score'); axes[1].set_ylabel('Count')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

---
## Part 2 — Why ROC-AUC Cannot Be Below 0.5

### Probabilistic Interpretation of AUC

**AUC has a beautiful closed-form meaning (Wilcoxon-Mann-Whitney statistic):**

$$\text{AUC} = P(\text{score}(x^+) > \text{score}(x^-))$$

where $x^+$ is a **randomly drawn positive** example and $x^-$ is a **randomly drawn negative** example.

> AUC is the probability that your classifier ranks a random positive higher than a random negative.

### The Symmetry Argument (Why AUC ≥ 0.5)

For any classifier with AUC < 0.5:

$$P(\text{score}(x^+) > \text{score}(x^-)) < 0.5$$

This means:

$$P(\text{score}(x^-) > \text{score}(x^+)) > 0.5$$

So the **negatives score higher than the positives** — the model has learned the task **backwards**.  
Simply **flipping all predictions** (1 → 0, 0 → 1) gives you a model with AUC > 0.5.

Therefore:
- **AUC = 1.0** → Perfect classifier
- **AUC = 0.5** → Worthless (random guessing)
- **AUC < 0.5** → Impossible in practice (just flip predictions)
- **AUC = 0.0** → Perfectly wrong (every positive ranked lower than every negative)

In [ ]:
# ── Verify the probabilistic interpretation numerically ───────────────

def auc_wilcoxon(y_true, scores):
    """
    Compute AUC via the Wilcoxon-Mann-Whitney U statistic:
    AUC = #{(i,j) : score_i > score_j, y_i=1, y_j=0} / (P * N)

    Completely independent of thresholds or integrals.
    """
    pos_scores = scores[y_true == 1]
    neg_scores = scores[y_true == 0]
    P = len(pos_scores)
    N = len(neg_scores)

    # Count pairs where positive scores higher than negative
    # Vectorised: for each positive, how many negatives does it beat?
    concordant = np.sum(
        pos_scores[:, np.newaxis] > neg_scores[np.newaxis, :]
    )
    # Ties count as 0.5
    ties = np.sum(
        pos_scores[:, np.newaxis] == neg_scores[np.newaxis, :]
    )
    return (concordant + 0.5 * ties) / (P * N)


print("AUC via trapezoid integration vs Wilcoxon statistic:")
print(f"{'Classifier':<22}  {'Trapezoid':>10}  {'Wilcoxon':>10}  {'Match?':>8}")
print("-" * 58)
for label, scores, _ in scenarios:
    fpr, tpr, _ = roc_curve_scratch(y_true, scores)
    auc_trap = auc_trapezoid(fpr, tpr)
    auc_wmw  = auc_wilcoxon(y_true, scores)
    match = "✓" if abs(auc_trap - auc_wmw) < 0.005 else "✗"
    print(f"{label:<22}  {auc_trap:>10.4f}  {auc_wmw:>10.4f}  {match:>8}")

print()
print("── What happens when we flip the bad classifier? ──────────────")
fpr_bad,  tpr_bad,  _ = roc_curve_scratch(y_true, scores_bad)
fpr_flip, tpr_flip, _ = roc_curve_scratch(y_true, 1 - scores_bad)  # flip!
auc_bad  = auc_trapezoid(fpr_bad,  tpr_bad)
auc_flip = auc_trapezoid(fpr_flip, tpr_flip)
print(f"Bad classifier AUC    : {auc_bad:.4f}")
print(f"Flipped classifier AUC: {auc_flip:.4f}")
print(f"Sum                   : {auc_bad + auc_flip:.4f}  ← always sums to 1.0")

In [ ]:
# ── Visual proof: bad classifier = mirror image of good classifier ────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Bad classifier ROC
fpr_bad, tpr_bad, _ = roc_curve_scratch(y_true, scores_bad)
axes[0].plot(fpr_bad, tpr_bad, color='crimson', linewidth=2.5, label=f'Bad  AUC={auc_bad:.3f}')
axes[0].plot([0,1],[0,1],'k--',linewidth=1)
axes[0].fill_between(fpr_bad, tpr_bad, np.linspace(0,1,len(fpr_bad)), alpha=0.15, color='crimson')
axes[0].set_title('Bad Classifier'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(); axes[0].set_xlim(0,1); axes[0].set_ylim(0,1)

# Plot 2: Flipped = mirror about diagonal
axes[1].plot(fpr_flip, tpr_flip, color='steelblue', linewidth=2.5, label=f'Flipped (1-score)  AUC={auc_flip:.3f}')
axes[1].plot([0,1],[0,1],'k--',linewidth=1)
axes[1].fill_between(fpr_flip, tpr_flip, np.linspace(0,1,len(fpr_flip)), alpha=0.15, color='steelblue')
axes[1].set_title('Flipped Classifier'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(); axes[1].set_xlim(0,1); axes[1].set_ylim(0,1)

# Plot 3: AUC vs. amount of noise in classifier
noise_levels = np.linspace(0, 1, 50)
aucs = []
for noise in noise_levels:
    # Interpolate between good and random classifier
    s = (1 - noise) * scores_good + noise * scores_random
    fpr_, tpr_, _ = roc_curve_scratch(y_true, s)
    aucs.append(auc_trapezoid(fpr_, tpr_))

axes[2].plot(noise_levels, aucs, color='teal', linewidth=2.5)
axes[2].axhline(0.5, color='gray', linestyle='--', linewidth=1.5, label='Random baseline (0.5)')
axes[2].set_title('AUC vs. Noise Level (Good → Random)')
axes[2].set_xlabel('Fraction of random noise added')
axes[2].set_ylabel('AUC')
axes[2].set_ylim(0.4, 1.0); axes[2].legend()

plt.suptitle('AUC < 0.5 = Backwards Model — Flip Scores, Get AUC > 0.5', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print()
print("Key Insight: The bad classifier's ROC curve is a reflection of")
print("the good classifier about the y=x diagonal.")
print("AUC(bad) + AUC(flipped) = 1.0  ← always true.")

---
## Part 3 — The Precision-Recall Curve

**PR curve** — plots Precision (y-axis) vs Recall (x-axis) as threshold decreases.

```
τ high → predict few positives → high precision (careful), low recall (misses many)
τ low  → predict many positives → low precision (noisy), high recall (catches most)
```

### Key Formulas

$$\text{Recall}    = \frac{TP}{TP + FN}  \qquad \text{(how many positives did we find?)}$$

$$\text{Precision} = \frac{TP}{TP + FP}  \qquad \text{(of what we found, how many are real?)}$$

### The Baseline

Unlike ROC whose random baseline is always a diagonal, the PR random baseline depends on **class prevalence**:

$$\text{Precision}_{\text{random}} = \frac{P}{P + N} = \text{prevalence}$$

A horizontal line at the prevalence rate. This makes PR far more sensitive to class imbalance.

In [ ]:
# ── PR curves for the same three classifiers ──────────────────────────

prevalence = y_true.mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Average Precision = area under PR curve (correct formula: sum of
# precision × Δrecall, which weights high-precision regions more)
def average_precision(recalls, precisions):
    """AP = Σ (recall[n] - recall[n-1]) × precision[n]"""
    order = np.argsort(recalls)
    r, p = recalls[order], precisions[order]
    return float(np.sum((r[1:] - r[:-1]) * p[1:]))

for label, scores, color in scenarios:
    rec, prec, _ = pr_curve_scratch(y_true, scores)
    ap = average_precision(rec, prec)
    # Sort by recall for a clean curve
    order = np.argsort(rec)
    axes[0].plot(rec[order], prec[order], color=color, linewidth=2.5, label=f'{label}  AP={ap:.3f}')

axes[0].axhline(prevalence, color='black', linestyle='--', linewidth=1.5,
                label=f'Random baseline (prevalence = {prevalence:.2f})')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curves')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.05)

# Precision-Recall tradeoff at different thresholds (good classifier)
thresholds_viz = np.linspace(0.1, 0.9, 50)
precs_viz, recs_viz, f1s_viz = [], [], []
for tau in thresholds_viz:
    TP, FP, TN, FN = confusion_at_threshold(y_true, scores_good, tau)
    p = TP / (TP + FP) if (TP + FP) > 0 else 1.0
    r = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    precs_viz.append(p); recs_viz.append(r); f1s_viz.append(f1)

axes[1].plot(thresholds_viz, precs_viz, color='steelblue', linewidth=2, label='Precision')
axes[1].plot(thresholds_viz, recs_viz,  color='crimson',   linewidth=2, label='Recall')
axes[1].plot(thresholds_viz, f1s_viz,   color='darkgreen', linewidth=2, label='F1 Score', linestyle='--')
best_f1_idx = np.argmax(f1s_viz)
axes[1].axvline(thresholds_viz[best_f1_idx], color='gray', linestyle=':', linewidth=1.5,
                label=f'Best F1 threshold = {thresholds_viz[best_f1_idx]:.2f}')
axes[1].set_xlabel('Threshold τ'); axes[1].set_ylabel('Metric value')
axes[1].set_title('Precision–Recall Trade-off vs Threshold (Good Classifier)')
axes[1].legend()

plt.tight_layout(); plt.show()
print(f'Optimal F1 threshold: {thresholds_viz[best_f1_idx]:.2f}')
print(f'At that threshold — Precision: {precs_viz[best_f1_idx]:.3f}  Recall: {recs_viz[best_f1_idx]:.3f}  F1: {f1s_viz[best_f1_idx]:.3f}')

---
## Part 4 — The Critical Difference: Class Imbalance

This is where the choice of ROC vs PR fundamentally matters.

### Why ROC Is Blind to Imbalance

FPR uses **TN** in its denominator:

$$\text{FPR} = \frac{FP}{FP + TN}$$

When negatives vastly outnumber positives (e.g. fraud: 0.1% of transactions),  
**TN is enormous**. Even if FP is large in absolute terms, FPR stays small —  
the model looks great on ROC.

Precision uses **FP** in its denominator without any TN:

$$\text{Precision} = \frac{TP}{TP + FP}$$

A high FP count directly crushes precision — the PR curve immediately shows the problem.

In [ ]:
# ── Simulate imbalanced dataset: 2% positives (like fraud detection) ──
rng2 = np.random.default_rng(7)

N_total = 5000
prevalences = [0.50, 0.20, 0.05, 0.01]   # 50%, 20%, 5%, 1% positive rate

fig = plt.figure(figsize=(18, 12))
gs  = GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.35)

for col, prev in enumerate(prevalences):
    n_pos = int(N_total * prev)
    n_neg = N_total - n_pos

    y = np.concatenate([np.ones(n_pos), np.zeros(n_neg)])
    scores = np.concatenate([
        rng2.normal(0.65, 0.18, n_pos),
        rng2.normal(0.35, 0.18, n_neg),
    ]).clip(0, 1)

    fpr,  tpr,  _ = roc_curve_scratch(y, scores)
    rec,  prec, _ = pr_curve_scratch(y, scores)

    roc_auc = auc_trapezoid(fpr, tpr)
    pr_ap   = average_precision(rec, prec)

    # ROC subplot
    ax_roc = fig.add_subplot(gs[0, col])
    ax_roc.plot(fpr, tpr, color='steelblue', linewidth=2)
    ax_roc.plot([0,1],[0,1],'k--',linewidth=1)
    ax_roc.set_title(f'ROC  (prev={prev:.0%})\nAUC = {roc_auc:.3f}', fontsize=10)
    ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
    ax_roc.set_xlim(0,1); ax_roc.set_ylim(0,1)
    ax_roc.fill_between(fpr, 0, tpr, alpha=0.15, color='steelblue')

    # PR subplot
    ax_pr = fig.add_subplot(gs[1, col])
    order = np.argsort(rec)
    ax_pr.plot(rec[order], prec[order], color='crimson', linewidth=2)
    ax_pr.axhline(prev, color='gray', linestyle='--', linewidth=1.5)
    ax_pr.set_title(f'PR  (prev={prev:.0%})\nAP = {pr_ap:.3f}', fontsize=10)
    ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
    ax_pr.set_xlim(0,1); ax_pr.set_ylim(0,1.05)
    ax_pr.fill_between(rec[order], 0, prec[order], alpha=0.15, color='crimson')
    ax_pr.text(0.5, prev + 0.03, 'Random baseline', ha='center', fontsize=8, color='gray')

plt.suptitle(
    'Same Classifier, Same Separation — Different Class Prevalence\n'
    'ROC stays high even at 1% prevalence; PR collapses and reveals true difficulty',
    fontsize=13, fontweight='bold'
)
plt.show()

print()
print(f'{"Prevalence":>12}  {"ROC-AUC":>10}  {"PR-AP":>10}')
print('-' * 36)
for prev in prevalences:
    n_pos = int(N_total * prev)
    n_neg = N_total - n_pos
    y = np.concatenate([np.ones(n_pos), np.zeros(n_neg)])
    scores = np.concatenate([
        rng2.normal(0.65, 0.18, n_pos),
        rng2.normal(0.35, 0.18, n_neg),
    ]).clip(0, 1)
    fpr_, tpr_, _ = roc_curve_scratch(y, scores)
    rec_, prec_, _ = pr_curve_scratch(y, scores)
    print(f'{prev:>12.0%}  {auc_trapezoid(fpr_,tpr_):>10.4f}  {average_precision(rec_,prec_):>10.4f}')
print()
print('ROC-AUC barely changes. PR-AP falls sharply at low prevalence → much more informative.')

---
## Part 5 — Why ROC Is Optimistic on Imbalanced Data: The Math

Imagine a fraud dataset: **10,000 transactions, 100 fraud (1%)**.  
A weak model flags 500 transactions as fraud, catching 80 true frauds.

```
TP = 80    FN = 20
FP = 420   TN = 9,480
```

### ROC perspective
$$\text{TPR} = \frac{80}{100} = 0.80  \qquad \text{FPR} = \frac{420}{9900} = 0.042$$

ROC operating point: **(0.042, 0.80)** — looks great! Near the top-left corner.

### PR perspective
$$\text{Precision} = \frac{80}{80 + 420} = \frac{80}{500} = 0.16  \qquad \text{Recall} = 0.80$$

PR operating point: **(0.80, 0.16)** — terrible! 84% of alerted transactions are false alarms.

**The TN = 9,480 washes out the FPR, hiding how noisy the model is.**  
PR has no TN anywhere — it directly exposes the noise.

In [ ]:
# ── Fraud detection: concrete numerical example ───────────────────────

TP, FP, TN, FN = 80, 420, 9480, 20
total = TP + FP + TN + FN
P = TP + FN
N = TN + FP

TPR       = TP / P
FPR       = FP / N
Precision = TP / (TP + FP)
Recall    = TPR
F1        = 2 * Precision * Recall / (Precision + Recall)
Accuracy  = (TP + TN) / total

print("=" * 55)
print("Fraud detection example: 10,000 transactions, 1% fraud")
print("Model flags 500 as fraud, catches 80 true frauds.")
print("=" * 55)
print(f"  Confusion:  TP={TP}  FP={FP}  TN={TN}  FN={FN}")
print()
print(f"  Accuracy    = {Accuracy:.3f}  ← 'great' (base rate is 99% anyway!)")
print(f"  ROC point   = FPR={FPR:.4f}, TPR={TPR:.3f}  ← looks good")
print(f"  PR  point   = Recall={Recall:.3f}, Precision={Precision:.3f}  ← only 16% of alerts are real fraud!")
print(f"  F1          = {F1:.3f}  ← operationalizes the bad precision")
print()
print(f"  Why FPR is low: FPR = FP/(FP+TN) = {FP}/({FP}+{TN}) = {FPR:.4f}")
print(f"  TN={TN} swamps FP={FP} → FPR looks tiny even though 420 false alarms!")

# Visualise the confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = np.array([[TP, FN], [FP, TN]])
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred Positive', 'Pred Negative'])
ax.set_yticklabels(['Actual Positive', 'Actual Negative'])
for i in range(2):
    for j in range(2):
        label = ['TP', 'FN', 'FP', 'TN'][i*2+j]
        ax.text(j, i, f'{label}\n{cm[i,j]}', ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_title('Fraud Detection Confusion Matrix', fontsize=12)
plt.tight_layout(); plt.show()

---
## Part 6 — How Each Curve Is Constructed Step by Step

In [ ]:
# Tiny example to trace through manually
y_tiny     = np.array([1, 0, 1, 0, 1, 0, 1, 0])
scores_tiny = np.array([0.9, 0.8, 0.7, 0.6, 0.55, 0.45, 0.3, 0.1])

P_t = y_tiny.sum()         # 4 positives
N_t = (1 - y_tiny).sum()   # 4 negatives

print(f"{'τ':>6}  {'TP':>4}  {'FP':>4}  {'TN':>4}  {'FN':>4}  {'TPR':>6}  {'FPR':>6}  {'Prec':>6}  {'Rec':>6}")
print("-" * 65)

rows = []
for tau in sorted(np.unique(scores_tiny), reverse=True):
    TP, FP, TN, FN = confusion_at_threshold(y_tiny, scores_tiny, tau)
    tpr  = TP / P_t
    fpr  = FP / N_t
    prec = TP / (TP + FP) if (TP + FP) > 0 else 1.0
    rec  = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    print(f"{tau:>6.2f}  {TP:>4}  {FP:>4}  {TN:>4}  {FN:>4}  {tpr:>6.3f}  {fpr:>6.3f}  {prec:>6.3f}  {rec:>6.3f}")
    rows.append((tau, tpr, fpr, prec, rec))

print("\nEach row is one point on the ROC and PR curves.")
print("As threshold decreases: more positives predicted → TPR↑ but also FPR↑")

# Plot the step-by-step curve construction
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fprs_t = [r[2] for r in rows]
tprs_t = [r[1] for r in rows]
precs_t = [r[3] for r in rows]
recs_t  = [r[4] for r in rows]
taus_t  = [r[0] for r in rows]

cmap = plt.cm.viridis(np.linspace(0, 1, len(rows)))

for i, (fpr_, tpr_, color) in enumerate(zip(fprs_t, tprs_t, cmap)):
    axes[0].scatter(fpr_, tpr_, color=color, s=120, zorder=5)
    axes[0].annotate(f'τ={taus_t[i]:.2f}', (fpr_, tpr_),
                     textcoords='offset points', xytext=(6, 2), fontsize=8)
axes[0].plot(fprs_t, tprs_t, 'gray', linewidth=1, alpha=0.4)
axes[0].plot([0,1],[0,1],'k--',linewidth=1)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve — Step by Step')
axes[0].set_xlim(-0.05, 1.05); axes[0].set_ylim(-0.05, 1.05)

for i, (rec_, prec_, color) in enumerate(zip(recs_t, precs_t, cmap)):
    axes[1].scatter(rec_, prec_, color=color, s=120, zorder=5)
    axes[1].annotate(f'τ={taus_t[i]:.2f}', (rec_, prec_),
                     textcoords='offset points', xytext=(6, 2), fontsize=8)
axes[1].plot(recs_t, precs_t, 'gray', linewidth=1, alpha=0.4)
axes[1].axhline(P_t/(P_t+N_t), color='k', linestyle='--', linewidth=1)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curve — Step by Step')
axes[1].set_xlim(-0.05, 1.05); axes[1].set_ylim(-0.05, 1.05)

plt.suptitle('Threshold decreases left→right on ROC, right→left on PR', fontsize=12)
plt.tight_layout(); plt.show()

---
## Part 7 — ROC vs PR for Real Scenarios

In [ ]:
# ── Side-by-side comparison for 4 real-world scenarios ────────────────

def make_scenario(n, prevalence, separation, seed):
    rng_ = np.random.default_rng(seed)
    n_pos = int(n * prevalence)
    n_neg = n - n_pos
    y = np.concatenate([np.ones(n_pos), np.zeros(n_neg)])
    scores = np.concatenate([
        rng_.normal(0.5 + separation/2, 0.2, n_pos),
        rng_.normal(0.5 - separation/2, 0.2, n_neg),
    ]).clip(0, 1)
    return y, scores

scenarios_real = {
    'Spam Detection\n(50% spam, good model)':     (2000, 0.50, 0.5, 1),
    'Disease Screening\n(10% prevalence)':        (2000, 0.10, 0.5, 2),
    'Fraud Detection\n(1% fraud, good model)':    (5000, 0.01, 0.6, 3),
    'Rare Disease\n(0.1%, hard to detect)':       (10000, 0.001, 0.4, 4),
}

fig, axes = plt.subplots(len(scenarios_real), 2, figsize=(14, 4*len(scenarios_real)))

for row, (name, (n, prev, sep, seed)) in enumerate(scenarios_real.items()):
    y, scores = make_scenario(n, prev, sep, seed)

    fpr, tpr, _   = roc_curve_scratch(y, scores)
    rec, prec, _  = pr_curve_scratch(y, scores)
    roc_auc = auc_trapezoid(fpr, tpr)
    pr_ap   = average_precision(rec, prec)

    order = np.argsort(rec)

    # ROC
    axes[row, 0].plot(fpr, tpr, 'steelblue', linewidth=2.5)
    axes[row, 0].plot([0,1],[0,1],'k--',linewidth=1)
    axes[row, 0].fill_between(fpr, 0, tpr, alpha=0.15, color='steelblue')
    axes[row, 0].set_title(f'{name}\nROC-AUC = {roc_auc:.3f}', fontsize=9)
    axes[row, 0].set_xlabel('FPR'); axes[row, 0].set_ylabel('TPR')
    axes[row, 0].set_xlim(0,1); axes[row, 0].set_ylim(0,1)
    axes[row, 0].text(0.6, 0.1, f'AUC={roc_auc:.3f}',
                      fontsize=11, fontweight='bold', color='steelblue')

    # PR
    axes[row, 1].plot(rec[order], prec[order], 'crimson', linewidth=2.5)
    axes[row, 1].axhline(prev, color='gray', linestyle='--', linewidth=1.5)
    axes[row, 1].fill_between(rec[order], 0, prec[order], alpha=0.15, color='crimson')
    axes[row, 1].set_title(f'{name}\nAvg Precision = {pr_ap:.3f}', fontsize=9)
    axes[row, 1].set_xlabel('Recall'); axes[row, 1].set_ylabel('Precision')
    axes[row, 1].set_xlim(0,1); axes[row, 1].set_ylim(0,1.05)
    axes[row, 1].text(0.5, 0.1 + prev, f'Random={prev:.3f}',
                      fontsize=9, color='gray', ha='center')
    axes[row, 1].text(0.1, 0.9, f'AP={pr_ap:.3f}',
                      fontsize=11, fontweight='bold', color='crimson')

plt.suptitle('ROC vs PR — Same Model, Different Class Imbalance Levels',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Part 8 — Summary: When to Use Which


In [ ]:
summary = [
    ("Situation",                     "Use ROC",     "Use PR"),
    ("-" * 25,                        "-" * 35,      "-" * 35),
    ("Balanced classes",              "✓ Standard",  "✓ Also fine"),
    ("Imbalanced (fraud, disease)",   "✗ Misleading","✓ Required"),
    ("Care about TN (spam filter)",   "✓ FPR encodes TN cost", "✗ TN invisible"),
    ("Care only about positives",     "✗ TN distorts", "✓ Directly relevant"),
    ("Comparing two models overall",  "✓ AUC summary", "✓ AP summary"),
    ("Choosing a threshold",          "✓ Pick operating point", "✓ Pick P/R tradeoff"),
    ("Reporting to clinicians",       "✗ Confusing",  "✓ Precision/Recall intuitive"),
    ("Ranking tasks (search)",        "✓ AUC = ranking prob", "✓ mAP for ranking"),
]
print(f"{'Situation':<26}  {'Use ROC':<35}  {'Use PR'}")
print("-" * 100)
for row in summary[2:]:
    print(f"{row[0]:<26}  {row[1]:<35}  {row[2]}")

print()
print("=" * 60)
print("Quick decision rule:")
print("  prevalence < ~10%  →  always use PR curve")
print("  prevalence ≥ ~30%  →  ROC and PR both informative")
print("  need to report cost of false alarms → PR Precision")
print("  need to report cost of missed detections → Recall")
print("=" * 60)

---
## Full Summary

### ROC Curve
- Plots **TPR vs FPR** as threshold sweeps 1 → 0
- AUC = P(score(positive) > score(negative)) — a **ranking probability**
- **Random baseline is always the diagonal** (AUC = 0.5), regardless of class ratio
- **Cannot be below 0.5**: an AUC of 0.3 means the model is backwards — flip predictions to get AUC = 0.7. Minimum useful AUC is 0.5 (random)
- **Weakness**: FPR's denominator includes TN. With many negatives, FPR looks tiny even when FP is large → ROC stays high even for a noisy classifier on imbalanced data

### PR Curve
- Plots **Precision vs Recall** as threshold sweeps
- AP = area under PR curve (weighted by Δrecall)
- **Random baseline = prevalence** (depends on class ratio, not fixed at 0.5)
- **Strength**: no TN anywhere — directly measures "when you flag something, how often are you right?"
- **Use when**: positives are rare, false alarms are costly, or you only care about finding the positive class

### The Fundamental Tension

```
Raising threshold τ:
  Precision ↑   (fewer, more confident predictions)
  Recall ↓      (miss more positives)

Lowering threshold τ:
  Recall ↑      (catch more positives)
  Precision ↓   (more false alarms)
```

F1 score = harmonic mean of Precision and Recall — a single-number summary of the PR tradeoff.  
The optimal F1 threshold is where the P and R curves cross.